<a href="https://colab.research.google.com/github/jaumg2004/xGMobile/blob/main/projeto_xGMobile.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import re
import unicodedata
from abc import ABC, abstractmethod
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# Projeto final de Python-xGMobile

## Objetivo do Projeto

Este projeto tem como objetivo desenvolver um pipeline robusto de **extração, limpeza, transformação e validação de dados**, partindo de um conjunto de dados brutos criado manualmente e contendo imperfeições naturais. Entre essas imperfeições estão espaços extras, acentuação, variações de caixa, formatos inconsistentes e valores numéricos representados como strings.

A proposta do SASD é simular um cenário real de Engenharia de Dados, no qual os dados raramente chegam prontos para uso analítico. Por isso, este notebook implementa uma arquitetura orientada a objetos para processar os dados brutos, corrigir inconsistências, validar o esquema e consolidar a saída em um **DataFrame do Pandas**, preparado para integração com bibliotecas de análise e Inteligência Artificial.

## Conjunto de Dados Brutos

O conjunto de dados utilizado neste projeto foi criado manualmente para atender à exigência de originalidade e autonomia analítica. Cada registro representa uma empresa e contém atributos textuais e numéricos relacionados a receita, retorno sobre investimento em IA e nível de maturidade em IA.

Os dados foram propositalmente construídos com inconsistências típicas de dados reais, tais como:

- espaços em branco no início e no fim de strings;
- uso inconsistente de letras maiúsculas e minúsculas;
- presença de acentos;
- percentuais representados como texto;
- separadores decimais diferentes, como ponto e vírgula;
- valores fora do intervalo permitido para teste de validação;
- diferenças de escrita para a mesma categoria.

Essas imperfeições servem como base para demonstrar as etapas de saneamento implementadas no pipeline.

## Arquitetura da Solução

A solução foi estruturada de forma modular para tornar o código mais legível, reutilizável e escalável. Em vez de concentrar toda a lógica em um único bloco, o pipeline foi dividido em componentes com responsabilidades específicas.

A arquitetura contém:

- um módulo de **exceções personalizadas**, responsável por representar erros de validação;
- um módulo de **descritores**, usado para controlar a qualidade de atributos críticos;
- uma **classe abstrata base**, que define a interface comum dos saneadores;
- subclasses especializadas para limpeza textual, transformação numérica e validação de esquema;
- uma classe principal de pipeline, responsável por coordenar a execução das etapas e retornar o DataFrame final.

Essa organização segue o princípio de separação de responsabilidades e atende à proposta de construção de um pipeline modular e orientado a objetos.

In [3]:
data = [
    {
        "company": "  Amazôn  ",
        "industry": "e-COMMERCE ",
        "country": " brasil ",
        "revenue_usd": " 453546400000 ",
        "ai_roi_percent": "30.33%",
        "ai_maturity_score": "77",
        "user_count": "310000000"
    },
    {
        "company": "OpenAI",
        "industry": " Technology",
        "country": "USA",
        "revenue_usd": "180000000",
        "ai_roi_percent": "19,5%",
        "ai_maturity_score": "100",
        "user_count": "400000000"
    },
    {
        "company": "  Microsoft ",
        "industry": "TECHNOLOGY",
        "country": " united states ",
        "revenue_usd": "211915000000",
        "ai_roi_percent": "28%",
        "ai_maturity_score": "95",
        "user_count": "1400000000"
    },
    {
        "company": "Goógle  ",
        "industry": " technology ",
        "country": "USA ",
        "revenue_usd": "307394000000 ",
        "ai_roi_percent": "26,7%",
        "ai_maturity_score": "91",
        "user_count": "4500000000"
    },
    {
        "company": "  Nubank",
        "industry": " FinTech ",
        "country": "Brasil",
        "revenue_usd": "8000000000",
        "ai_roi_percent": "18%",
        "ai_maturity_score": "84",
        "user_count": "100000000"
    },
    {
        "company": "Teslá ",
        "industry": " Automotive",
        "country": " usa",
        "revenue_usd": "96773000000",
        "ai_roi_percent": "22.4%",
        "ai_maturity_score": "89",
        "user_count": "5000000"
    },
    {
        "company": "  Samsung",
        "industry": "electronics ",
        "country": " south korea ",
        "revenue_usd": "200000000000",
        "ai_roi_percent": "17,2%",
        "ai_maturity_score": "82",
        "user_count": "1200000000"
    },
    {
        "company": "Mercadó Livre",
        "industry": " E-commerce",
        "country": " argentina ",
        "revenue_usd": "14800000000 ",
        "ai_roi_percent": "24%",
        "ai_maturity_score": "86",
        "user_count": "218000000"
    },
    {
        "company": "  Petrobras ",
        "industry": "Energy ",
        "country": " BRASIL",
        "revenue_usd": "124474000000",
        "ai_roi_percent": "12,8%",
        "ai_maturity_score": "73",
        "user_count": "30000000"
    },
    {
        "company": "IBM",
        "industry": "Technology  ",
        "country": "usa",
        "revenue_usd": "61860000000",
        "ai_roi_percent": "15%",
        "ai_maturity_score": "88",
        "user_count": "250000000"
    },
    {
        "company": "  ifood",
        "industry": "food tech ",
        "country": "Brasil ",
        "revenue_usd": "2200000000",
        "ai_roi_percent": "21,1%",
        "ai_maturity_score": "79",
        "user_count": "55000000"
    },
    {
        "company": "AliBába",
        "industry": "E-COMMERCE",
        "country": " china ",
        "revenue_usd": "126491000000",
        "ai_roi_percent": "23%",
        "ai_maturity_score": "90",
        "user_count": "903000000"
    },
    {
        "company": "  Siemens ",
        "industry": "industrial automation",
        "country": " germany",
        "revenue_usd": "83000000000 ",
        "ai_roi_percent": "14.6%",
        "ai_maturity_score": "81",
        "user_count": "385000000"
    },
    {
        "company": "Spotify ",
        "industry": " MediaTech ",
        "country": " sweden ",
        "revenue_usd": "15000000000",
        "ai_roi_percent": "16%",
        "ai_maturity_score": "76",
        "user_count": "626000000"
    },
    {
        "company": "  ByteDance",
        "industry": "technology",
        "country": "China ",
        "revenue_usd": "110000000000",
        "ai_roi_percent": "27,3%",
        "ai_maturity_score": "93",
        "user_count": "1500000000"
    },
    {
        "company": "Magalu",
        "industry": " retail ",
        "country": " brasil ",
        "revenue_usd": "35000000000",
        "ai_roi_percent": "11%",
        "ai_maturity_score": "72",
        "user_count": "35000000"
    }
]

## Etapa 1 — Extração e Limpeza Textual

Na primeira etapa do pipeline, os dados brutos são carregados a partir de uma estrutura do tipo **lista de dicionários**, em que cada dicionário representa um registro.

Em seguida, é aplicada a limpeza textual dos campos categóricos, com foco em:

- remoção de espaços excedentes;
- conversão para um padrão consistente de caixa;
- normalização de acentos;
- redução de múltiplos espaços internos;
- padronização de nomes de categorias.

Essa etapa é importante para evitar que valores semanticamente iguais sejam tratados como diferentes, como por exemplo `"Brasil"`, `" brasil "` e `"BRASIL"`.

In [4]:
class SanitizationError(Exception):
    """Erro base do processo de saneamento."""
    pass

class SchemaValidationError(SanitizationError):
    """Erro lançado quando um campo obrigatório está ausente ou inválido."""
    pass

## Etapa 2 — Transformação Numérica

Após a normalização textual, os campos numéricos são processados para conversão em tipos adequados.

Os principais tratamentos realizados são:

- remoção do símbolo de porcentagem;
- substituição de vírgula por ponto em valores decimais;
- conversão de strings numéricas para `float`;
- preparação dos dados para cálculos posteriores e uso em bibliotecas de análise.

Essa etapa garante que atributos como receita e retorno sobre investimento possam ser tratados de forma matemática e vetorizada, sem depender de correções manuais futuras.

In [5]:
class ScoreDescriptor:
    """Valida se um atributo numérico está entre 0 e 100."""

    def __set_name__(self, owner, name):
        self.private_name = "_" + name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return getattr(instance, self.private_name, None)

    def __set__(self, instance, value):
        if not (0 <= value <= 100):
            raise ValueError(f"{self.private_name} must be between 0 and 100.")
        setattr(instance, self.private_name, value)

## Etapa 3 — Validação e Controle de Qualidade

Além da limpeza e transformação, o pipeline também realiza validações estruturais e semânticas.

A validação estrutural verifica se cada registro contém os campos obrigatórios esperados pelo sistema. Já a validação semântica assegura que determinados valores estejam dentro de faixas aceitáveis.

Para isso, foi utilizado um **descritor** para validar o atributo `ai_maturity_score`, exigindo que ele esteja no intervalo de 0 a 100. Caso um valor inválido seja fornecido, o método de atribuição lança `ValueError`, como exigido pela especificação do projeto.

Essa abordagem fortalece o controle de qualidade dos dados e evita que registros inconsistentes avancem para etapas posteriores do pipeline.

In [6]:
class RecordModel:
    """Modelo simples para validar score com descritor."""
    ai_maturity_score = ScoreDescriptor()

    def __init__(self, ai_maturity_score):
        self.ai_maturity_score = ai_maturity_score

## Uso de Programação Orientada a Objetos

A implementação do pipeline utiliza conceitos de Programação Orientada a Objetos para tornar a solução extensível e organizada.

Os principais conceitos aplicados foram:

- **classe abstrata (`ABC`)** para definir a interface comum dos saneadores;
- **herança**, permitindo que saneadores especializados reutilizem a lógica da classe base;
- **polimorfismo**, por meio da redefinição do método `clean()` nas subclasses;
- uso de **`super()`** para aproveitar o comportamento herdado e expandi-lo;
- uso de **`@staticmethod`** para funções utilitárias independentes do estado da instância;
- uso de **`@classmethod`** para metadados e configuração da classe.

Essa modelagem permite adicionar novas regras de saneamento no futuro sem reescrever toda a estrutura do sistema.

In [7]:
class BaseSanitizer(ABC):
    """Interface padrão dos saneadores do SASD."""

    def __init__(self, records):
        self.records = records

    @abstractmethod
    def calibrate(self):
        """Define regras de saneamento."""
        pass

    @abstractmethod
    def clean(self):
        """Aplica a limpeza aos dados."""
        pass

In [8]:
class TextSanitizer(BaseSanitizer):
    """Saneador de campos textuais."""

    def calibrate(self):
        self.text_fields = ["company", "industry", "country"]

    @staticmethod
    def normalize_text(value):
        value = str(value).strip().lower()
        value = unicodedata.normalize("NFKD", value).encode("ascii", "ignore").decode("utf-8")
        return re.sub(r"\s+", " ", value)

    def clean(self):
        self.calibrate()
        cleaned = []
        for record in self.records:
            new_record = record.copy()
            for field in self.text_fields:
                if field in new_record:
                    new_record[field] = self.normalize_text(new_record[field])
            cleaned.append(new_record)
        return cleaned


## Tratamento de Exceções

Para aumentar a robustez do pipeline, foram utilizadas exceções personalizadas com `try/except`.

Quando um valor numérico inválido ou um campo inconsistente é identificado, a exceção original é capturada e convertida em uma exceção específica do processo de saneamento. Isso melhora a clareza do erro e facilita a depuração do sistema.

Essa estratégia torna o código mais resiliente e mais próximo de um cenário real de processamento de dados, no qual falhas precisam ser tratadas de forma controlada e informativa.

In [9]:
class NumericSanitizer(TextSanitizer):
    """Saneador numérico especializado."""

    def calibrate(self):
        super().calibrate()
        self.numeric_fields = ["revenue_usd",
            "ai_roi_percent",
            "ai_maturity_score",
            "user_count"]

    @staticmethod
    def parse_number(value):
        value = str(value).strip().replace("%", "").replace(",", ".")
        return float(value)

    @classmethod
    def required_fields(cls):
        return ["company", "industry", "country", "revenue_usd", "ai_roi_percent", "ai_maturity_score", "user_count"]

    def clean(self):
        records = super().clean()
        self.calibrate()

        cleaned = []
        for record in records:
            new_record = record.copy()
            try:
                for field in self.numeric_fields:
                    new_record[field] = self.parse_number(new_record[field])

                model = RecordModel(new_record["ai_maturity_score"])
                new_record["ai_maturity_score"] = model.ai_maturity_score

                cleaned.append(new_record)
            except ValueError as exc:
                raise SchemaValidationError(f"Invalid numeric value in record: {record}") from exc

        return cleaned


In [10]:
class SchemaValidator(NumericSanitizer):
    """Valida a presença dos campos obrigatórios."""

    def clean(self):
        records = super().clean()
        self.calibrate()

        cleaned = []
        invalid_records = []

        for record in records:
            new_record = record.copy()
            try:
                for field in self.numeric_fields:
                    new_record[field] = self.parse_number(new_record[field])

                model = RecordModel(new_record["ai_maturity_score"])
                new_record["ai_maturity_score"] = model.ai_maturity_score

                cleaned.append(new_record)

            except ValueError as exc:
                invalid_records.append({
                    "record": record,
                    "error": str(exc)
                })

        print("Invalid records found:")
        for item in invalid_records:
            print(item)

        return cleaned

## Execução do Pipeline

A execução principal do sistema ocorre por meio de uma classe orquestradora responsável por:

1. receber os dados brutos;
2. instanciar o saneador adequado;
3. aplicar as etapas de limpeza, transformação e validação;
4. consolidar os registros válidos em um DataFrame final.

Ao final do processo, o conjunto de dados encontra-se padronizado, validado e pronto para análises quantitativas ou integração com modelos de aprendizado de máquina.

In [11]:
class SASDPipeline:
    """Executa o pipeline completo de saneamento."""

    def __init__(self, data):
        self.data = data

    def run(self):
        sanitizer = SchemaValidator(self.data)
        cleaned_records = sanitizer.clean()
        return pd.DataFrame(cleaned_records)

In [12]:
pipeline = SASDPipeline(data)
df_final = pipeline.run()
df_final

Invalid records found:


,company,industry,country,revenue_usd,ai_roi_percent,ai_maturity_score,user_count
0,amazon,e-commerce,brasil,4.535464e+11,30.33,77.0,3.100000e+08
1,openai,technology,usa,1.800000e+08,19.50,100.0,4.000000e+08
2,microsoft,technology,united states,2.119150e+11,28.00,95.0,1.400000e+09
3,google,technology,usa,3.073940e+11,26.70,91.0,4.500000e+09
4,nubank,fintech,brasil,8.000000e+09,18.00,84.0,1.000000e+08
5,tesla,automotive,usa,9.677300e+10,22.40,89.0,5.000000e+06
6,samsung,electronics,south korea,2.000000e+11,17.20,82.0,1.200000e+09
7,mercado livre,e-commerce,argentina,1.480000e+10,24.00,86.0,2.180000e+08
8,petrobras,energy,brasil,1.244740e+11,12.80,73.0,3.000000e+07
9,ibm,technology,usa,6.186000e+10,15.00,88.0,2.500000e+08


## Integração com Pandas, NumPy e Scikit-Learn

Após o saneamento, os dados são convertidos para um **DataFrame do Pandas**, permitindo manipulação tabular eficiente e definição consistente de tipos de dados.

Essa saída final é compatível com o ecossistema de IA, especialmente com:

- **NumPy**, para operações vetorizadas e processamento numérico;
- **Scikit-Learn**, para escalonamento, separação de treino e teste, classificação, regressão e validação de modelos.

Dessa forma, o SASD não apenas limpa os dados, mas também entrega uma base pronta para fluxos posteriores de análise estatística e modelagem preditiva.

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_final[["revenue_usd", "ai_roi_percent", "user_count"]].to_numpy()
y = df_final["ai_maturity_score"].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)